In [7]:
src_lang = "ind"
target_lang = ["aaz", "ptu", "nfa", "heg", "lex", "row", "llg", "rgu", "txq", "tet", "wrs"]
NT_BOOKS = [
    "MAT",
    "MRK",
    "LUK",
    "JHN",
    "ACT",
    "ROM",
    "1CO",
    "2CO",
    "GAL",
    "EPH",
    "PHP",
    "COL",
    "1TH",
    "2TH",
    "1TI",
    "2TI",
    "TIT",
    "PHM",
    "HEB",
    "JAS",
    "1PE",
    "2PE",
    "1JN",
    "2JN",
    "3JN",
    "JUD",
    "REV",
]

In [1]:
from datasets import load_dataset

dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=["ind", "aaz"], trust_remote_code=True)

print(dataset)

/home/bookbot/miniconda3/envs/nmt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 7102
    })
    validation: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 385
    })
    test: Dataset({
        features: ['translation', 'files', 'ref', 'licenses', 'copyrights'],
        num_rows: 384
    })
})


In [13]:
from datasets import load_dataset, DatasetDict

def process_translations(x):
    """Process translations to handle multiple references and concatenate by language"""
    languages = x["translation"]["language"]
    translations = x["translation"]["translation"]
    
    # Group translations by language
    lang_translations = {}
    for lang, translation in zip(languages, translations):
        if lang not in lang_translations:
            lang_translations[lang] = []
        lang_translations[lang].append(translation)
    
    # Concatenate translations for each language with space separator
    src_text = " ".join(lang_translations.get(src_lang, [""]))
    tgt_text = " ".join(lang_translations.get(tgt_lang, [""]))
    
    return {"text_source": src_text, "text_target": tgt_text}
    

def load_ebible_corpus(src_lang, tgt_lang):
    dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=[src_lang, tgt_lang], trust_remote_code=True)
    dataset = dataset.map(process_translations)
    # OT books for testing, NT books for training and validation
    # Handle both single refs and multiple refs
    def is_nt_book(refs):
        if isinstance(refs, list):
            # Check if any ref belongs to NT books
            return any(ref.split()[0] in NT_BOOKS for ref in refs)
        else:
            # Single reference
            return refs.split()[0] in NT_BOOKS
    
    # The dataset is a DatasetDict, so we need to access the 'train' split
    train_data = dataset['train']
    train_ds = train_data.filter(lambda x: is_nt_book(x["ref"]))
    test_ds = train_data.filter(lambda x: not is_nt_book(x["ref"]))
    train_val_ds = train_ds.train_test_split(test_size=0.1, seed=41)
    dataset = DatasetDict({"train": train_val_ds["train"], "validation": train_val_ds["test"], "test": test_ds})
    return dataset

# Process all target languages and combine into one dataset
all_datasets = {}

for tgt_lang in target_lang:
    print(f"Processing {source_lang} -> {tgt_lang}...")
    dataset = load_ebible_corpus(source_lang, tgt_lang)

# Create subset names for each split
subset_name = f"{source_lang}-{tgt_lang}"

# Add each split with the language pair prefix
for split_name, split_data in dataset.items():
    full_subset_name = f"{subset_name}-{split_name}"
    all_datasets[full_subset_name] = split_data
    print(f"  Added {full_subset_name}: {len(split_data)} examples")

# Combine all into one DatasetDict
combined_dataset = DatasetDict(all_datasets)

print(f"\nFinal combined dataset structure:")
for subset_name, subset_data in combined_dataset.items():
    print(f"  {subset_name}: {len(subset_data)} examples")


Processing ind -> aaz...
Processing ind -> ptu...


100%|██████████| 18785/18785 [00:00<00:00, 128043.66it/s]
Generating train split: 7289 examples [00:00, 14206.00 examples/s]
Generating validation split: 398 examples [00:00, 11216.22 examples/s]
Generating test split: 392 examples [00:00, 11190.82 examples/s]
Filter: 100%|██████████| 7289/7289 [00:00<00:00, 81366.82 examples/s]


Processing ind -> nfa...


100%|██████████| 18785/18785 [00:00<00:00, 127798.17it/s]
Generating train split: 7097 examples [00:00, 13907.87 examples/s]
Generating validation split: 390 examples [00:00, 10812.56 examples/s]
Generating test split: 389 examples [00:00, 11048.40 examples/s]
Filter: 100%|██████████| 7097/7097 [00:00<00:00, 58173.62 examples/s]


Processing ind -> heg...


...

Processing ind -> tet...


Filter: 100%|██████████| 7097/7097 [00:00<00:00, 79855.18 examples/s]


Processing ind -> wrs...


100%|██████████| 18785/18785 [00:00<00:00, 172133.17it/s]
Generating train split: 6851 examples [00:00, 12850.42 examples/s]
Generating validation split: 382 examples [00:00, 10618.21 examples/s]
Generating test split: 379 examples [00:00, 10673.60 examples/s]
Filter: 100%|██████████| 6851/6851 [00:00<00:00, 82090.89 examples/s]

  Added ind-wrs-train: 6088 examples
  Added ind-wrs-validation: 677 examples
  Added ind-wrs-test: 86 examples

Final combined dataset structure:
  ind-wrs-train: 6088 examples
  ind-wrs-validation: 677 examples
  ind-wrs-test: 86 examples


In [14]:
# Push the combined dataset to Hugging Face Hub
def push_to_hub(dataset, repo_name, private=False):
    """
    Push the processed dataset to Hugging Face Hub
    
    Args:
        dataset: The DatasetDict to push
        repo_name: Name of the repository (e.g., "username/dataset-name")
        private: Whether to make the repository private
    """
    try:
        # Push the dataset
        dataset.push_to_hub(
            repo_id=repo_name,
            private=private,
            token=True  # Uses your saved HF token
        )
        print(f"✅ Successfully pushed dataset to: https://huggingface.co/datasets/{repo_name}")
        
        # Print dataset card information
        print(f"\n📝 Dataset structure:")
        for subset_name in dataset.keys():
            lang_pair = subset_name.rsplit('-', 1)[0]  # Remove split suffix
            split = subset_name.rsplit('-', 1)[1]      # Get split name
            print(f"  {subset_name}: {len(dataset[subset_name])} examples")
            
    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("Make sure you're logged in with `huggingface-cli login`")

# Example usage (uncomment and modify as needed):
# REPO_NAME = "your-username/bible-nmt-multilingual"  # Change this to your desired repo name
# push_to_hub(combined_dataset, REPO_NAME, private=False)

print("Dataset is ready to be pushed to Hub!")
print("Uncomment and modify the REPO_NAME above, then run the push_to_hub function.")


Dataset is ready to be pushed to Hub!
Uncomment and modify the REPO_NAME above, then run the push_to_hub function.


In [15]:
combined_dataset.push_to_hub("biblenlp-corpus")

ValueError: Split name should match '^\w+(\.\w+)*$' but got 'ind-wrs-train'.

In [46]:
for datum in dataset["train"]:
    if len(datum["ref"]) > 1:
        print(datum["ref"])
        print(datum["translation"]["language"])
        print(datum["translation"])
        print(len(datum["translation"]["translation"]))
        print(datum["text_source"])
        print(datum["text_target"])
        # break

['JHN 4:17', 'JHN 4:18']
['ind', 'ind', 'tet']
{'language': ['ind', 'ind', 'tet'], 'translation': ['karena Ibu sudah kawin lima kali dengan laki-laki yang berbeda-beda. Saat ini laki-laki yang hidup bersamamu juga bukanlah suamimu.”', 'karena kamu sudah lima kali kawin cerai dengan laki-laki yang berbeda. Dan laki-laki yang hidup bersamamu sekarang bukanlah suamimu. Ya, perkataanmu itu memang benar.”', 'Nia nataa naꞌak, “Mais haꞌu la koo laꞌen.” Yesus moos naꞌak, “Tebes. Bii dale moon. O moo laꞌen lima tiꞌan. Mane mak iha baa oras neꞌe moos, lahoos okaan laꞌen.”']}
3
karena Ibu sudah kawin lima kali dengan laki-laki yang berbeda-beda. Saat ini laki-laki yang hidup bersamamu juga bukanlah suamimu.” karena kamu sudah lima kali kawin cerai dengan laki-laki yang berbeda. Dan laki-laki yang hidup bersamamu sekarang bukanlah suamimu. Ya, perkataanmu itu memang benar.”
Nia nataa naꞌak, “Mais haꞌu la koo laꞌen.” Yesus moos naꞌak, “Tebes. Bii dale moon. O moo laꞌen lima tiꞌan. Mane mak iha baa 

In [2]:
from datasets import load_dataset

dataset = load_dataset("bible-nlp/biblenlp-corpus", languages=["ind", "aaz"], trust_remote_code=True)

print(dataset["train"][0])

{'translation': {'language': ['aaz', 'ind'], 'translation': ['Au ꞌtoit he Usif Yesus nakriraꞌ In nekan arekot neu Iin na ok-okeꞌ. Tebes namneo. Amin. Naꞌko au, Naiꞌ Yohanis kau, tua.', 'Semoga kasih karunia Tuhan Yesus bersama dengan orang-orang percaya. Amin.']}, 'files': {'lang': ['aaz', 'ind'], 'file': ['aaz-aaz.txt', 'ind-indags.txt']}, 'ref': ['REV 22:21'], 'licenses': ['http://creativecommons.org/licenses/by-nd/4.0/', 'http://creativecommons.org/licenses/by-sa/4.0/'], 'copyrights': ['Unit Bahasa dan Budaya, Kupang NTT, Indonesia', '']}
